# Notebook 3 — Feature Engineering

**Canonical training script:** `scripts/train.py`. This notebook **duplicates** Stage 4 (feature engineering) for teaching — when the script changes, update these cells to match.

**Dense features:** concatenation yields **109** dense columns (+ 45,000 TF-IDF from full + diff). `prompt_question_type` appears **only** in `align_features`, not duplicated in `prompt_comp` (matches production).

**TF-IDF:** same parameters as `train.py` (no `strip_accents`).


## Imports and optional libraries

Same graceful fallbacks as `train.py` (textstat, TextBlob, VADER, optional spaCy).

In [ ]:
import ast
import os
import re
import time
import warnings

warnings.filterwarnings("ignore")

import joblib
import numpy as np
import pandas as pd
import scipy.sparse as sp

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler

# Canonical pipeline: scripts/train.py — keep this notebook in sync manually.
# Set True to mirror CLI --enable-spacy (slow; needs en_core_web_sm).
RUN_SPACY = False

try:
    import textstat
    HAS_TEXTSTAT = True
except ImportError:
    HAS_TEXTSTAT = False

try:
    from textblob import TextBlob
    HAS_TEXTBLOB = True
except ImportError:
    HAS_TEXTBLOB = False

try:
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
    _vader = SentimentIntensityAnalyzer()
    HAS_VADER = True
except ImportError:
    HAS_VADER = False

try:
    import spacy
    if RUN_SPACY:
        _nlp = spacy.load("en_core_web_sm")
        HAS_SPACY = True
    else:
        _nlp = None
        HAS_SPACY = False
except Exception:
    _nlp = None
    HAS_SPACY = False

ARTIFACTS = "../artifacts"
os.makedirs(ARTIFACTS, exist_ok=True)

print("OPTIONAL LIBRARY STATUS")
print(f"  textstat   : {'OK' if HAS_TEXTSTAT else 'missing'}")
print(f"  textblob   : {'OK' if HAS_TEXTBLOB else 'missing'}")
print(f"  vader      : {'OK' if HAS_VADER else 'missing'}")
print(f"  spaCy NER  : {'OK' if HAS_SPACY else 'off (set RUN_SPACY=True)'}")

t0 = time.time()


## Text helpers and feature functions

Copied from `scripts/train.py` (stopwords through `prompt_to_response_ratio`).

In [ ]:
# ── STOPWORDS ─────────────────────────────────────────────────
STOPWORDS = {
    'a','an','the','and','or','but','in','on','at','to','for','of','with',
    'by','from','is','are','was','were','be','been','has','have','had',
    'do','does','did','will','would','could','should','may','might','shall',
    'this','that','these','those','i','you','he','she','it','we','they',
    'me','him','her','us','them','my','your','his','its','our','their',
    'what','which','who','how','when','where','why','not','no','so','as',
    'up','out','if','about','into','than','then','some','more','also',
    'just','like','can','all','there','been','being','very','get',
}

# ── Sentiment/tone word lists (no external lib needed) ────────
POSITIVE_WORDS = {
    'good','great','excellent','best','perfect','wonderful','amazing','fantastic',
    'helpful','clear','correct','accurate','right','easy','simple','effective',
    'well','better','improved','nice','useful','benefit','advantage','success',
    'recommend','love','appreciate','thank','happy','pleased','satisfied',
}
NEGATIVE_WORDS = {
    'bad','wrong','error','incorrect','fail','failure','problem','issue','bug',
    'difficult','hard','confusing','confused','unclear','terrible','awful',
    'poor','worse','broken','missing','lack','sorry','unfortunately','unable',
    'cannot','impossible','never','avoid','danger','risk','harm','worse',
}
HEDGE_WORDS = {
    'perhaps','maybe','might','could','possibly','somewhat','arguably','generally',
    'usually','often','sometimes','occasionally','approximately','roughly','about',
    'around','likely','unlikely','probably','presumably','apparently','seemingly',
    'suggest','seem','appear','tend','consider','believe','think','assume',
}
SYCOPHANCY_PHRASES = [
    "great question", "excellent question", "good question", "wonderful question",
    "of course", "certainly", "absolutely", "definitely", "sure thing",
    "i'd be happy to", "i'd be glad to", "i'm happy to help",
    "that's a great", "that's an excellent",
]
COMMON_5000_WORDS = STOPWORDS | {
    # a sample of very common words beyond stopwords
    'said','one','two','three','four','five','six','seven','eight','nine','ten',
    'time','year','people','way','day','man','woman','child','world','life',
    'hand','part','place','case','week','company','system','program','question',
    'work','government','number','night','point','home','water','room','mother',
    'area','money','story','fact','month','lot','right','study','book','eye',
    'job','word','business','issue','side','kind','head','house','service',
    'friend','father','power','hour','game','line','end','among','while','might',
    'next','sound','below','saw','something','thought','both','paper','use','together',
    'got','never','know','put','does','told','nothing','sure','come','few',
}

# ── HELPERS ───────────────────────────────────────────────────
def parse_field(val):
    """Parse JSON-list or plain string conversation field."""
    if pd.isna(val): return ''
    v = str(val).strip()
    if v.startswith('['):
        try: return ' '.join(str(p) for p in __import__('json').loads(v))
        except:
            try: return ' '.join(str(p) for p in ast.literal_eval(v))
            except: return v
    return v

def clean_text(text):
    """Full cleaning: lowercase → strip markdown → remove URLs → alpha only → remove stopwords."""
    if not isinstance(text, str) or text.strip() == '': return ''
    text = text.lower()
    text = re.sub(r'```[\s\S]{0,3000}?```', ' ', text)
    text = re.sub(r'`[^`]+`', ' ', text)
    text = re.sub(r'\*{1,3}([^*]{0,300})\*{1,3}', r'\1', text)
    text = re.sub(r'#{1,6}\s', ' ', text)
    text = re.sub(r'\[([^\]]+)\]\([^)]+\)', r'\1', text)
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return ' '.join(w for w in text.split() if w not in STOPWORDS and len(w) >= 2)

def get_diff(ta, tb):
    sa, sb = set(ta.split()), set(tb.split())
    return ' '.join(sorted(sa - sb)) + ' ' + ' '.join(sorted(sb - sa))

# ── Original helpers ───────────────────────────────────────────
def count_sentences(t): return max(1, len(re.split(r'[.!?]+', t))) if t else 0
def vocab_richness(t):
    tok = t.split(); return len(set(tok)) / len(tok) if tok else 0.0
def has_code(t): return int('```' in t)
def has_bullets(t): return int(bool(re.search(r'\n\s*[-•*]\s|\n\s*\d+\.\s', t)))
def has_headers(t): return int(bool(re.search(r'\n#{1,4}\s', t)))
def jaccard(a, b):
    sa, sb = set(a.split()), set(b.split())
    if not sa and not sb: return 1.0
    if not sa or not sb: return 0.0
    return len(sa & sb) / len(sa | sb)
def overlap_coeff(a, b):
    sa, sb = set(a.split()), set(b.split())
    if not sa or not sb: return 0.0
    return len(sa & sb) / min(len(sa), len(sb))
def bigram_overlap(a, b):
    def bg(t): tok = t.split(); return set(zip(tok[:-1], tok[1:]))
    ba, bb = bg(a), bg(b)
    if not ba or not bb: return 0.0
    return len(ba & bb) / (len(ba | bb) + 1e-9)

# ── NEW: Readability helpers ───────────────────────────────────
def fk_grade(text):
    """Flesch-Kincaid grade level. Requires textstat."""
    if HAS_TEXTSTAT and text.strip():
        try: return textstat.flesch_kincaid_grade(text)
        except: pass
    # Fallback: rough approximation
    words = text.split()
    sentences = max(1, count_sentences(text))
    syllables = sum(max(1, len(re.findall(r'[aeiou]', w.lower()))) for w in words)
    if not words: return 0.0
    return 0.39 * (len(words) / sentences) + 11.8 * (syllables / len(words)) - 15.59

def flesch_ease(text):
    """Flesch reading ease (0-100, higher = easier). Requires textstat."""
    if HAS_TEXTSTAT and text.strip():
        try: return textstat.flesch_reading_ease(text)
        except: pass
    # Fallback approximation
    words = text.split()
    sentences = max(1, count_sentences(text))
    syllables = sum(max(1, len(re.findall(r'[aeiou]', w.lower()))) for w in words)
    if not words: return 50.0
    return 206.835 - 1.015 * (len(words) / sentences) - 84.6 * (syllables / len(words))

def avg_syllables_per_word(text):
    words = [w for w in text.split() if w]
    if not words: return 0.0
    if HAS_TEXTSTAT:
        try: return textstat.avg_syllables_per_word(text)
        except: pass
    return sum(max(1, len(re.findall(r'[aeiou]', w.lower()))) for w in words) / len(words)

def avg_words_per_sentence(text):
    words = text.split()
    sentences = max(1, count_sentences(text))
    return len(words) / sentences if words else 0.0

# ── NEW: Structural formatting counts ─────────────────────────
def count_headers(t):
    return len(re.findall(r'\n#{1,6}\s', t))

def count_bullets(t):
    return len(re.findall(r'\n\s*[-•*]\s|\n\s*\d+\.\s', t))

def count_code_blocks(t):
    return len(re.findall(r'```', t)) // 2  # each block has open + close

def count_paragraphs(t):
    """Count double-newline separated paragraphs."""
    parts = re.split(r'\n\s*\n', t.strip())
    return max(1, len([p for p in parts if p.strip()]))

def formatting_richness(t):
    """Composite: headers + bullets + code blocks."""
    return count_headers(t) + count_bullets(t) + count_code_blocks(t)

# ── NEW: Sentiment / tone helpers ─────────────────────────────
def sentiment_polarity(text):
    """Sentiment polarity (-1 to +1). Uses VADER > TextBlob > word list."""
    if not text.strip(): return 0.0
    if HAS_VADER:
        try: return _vader.polarity_scores(text)['compound']
        except: pass
    if HAS_TEXTBLOB:
        try: return TextBlob(text).sentiment.polarity
        except: pass
    # Fallback: simple word-list ratio
    words = text.lower().split()
    if not words: return 0.0
    pos = sum(1 for w in words if w in POSITIVE_WORDS)
    neg = sum(1 for w in words if w in NEGATIVE_WORDS)
    return (pos - neg) / len(words)

def positive_word_ratio(clean_text_tokens):
    tok = clean_text_tokens.split()
    if not tok: return 0.0
    return sum(1 for w in tok if w in POSITIVE_WORDS) / len(tok)

def negative_word_ratio(clean_text_tokens):
    tok = clean_text_tokens.split()
    if not tok: return 0.0
    return sum(1 for w in tok if w in NEGATIVE_WORDS) / len(tok)

def hedge_word_ratio(clean_text_tokens):
    tok = clean_text_tokens.split()
    if not tok: return 0.0
    return sum(1 for w in tok if w in HEDGE_WORDS) / len(tok)

def sycophancy_flag(raw_text):
    """1 if response opens with sycophantic phrase."""
    first_100 = raw_text.lower()[:100]
    return int(any(p in first_100 for p in SYCOPHANCY_PHRASES))

# ── NEW: Prompt-response alignment helpers ────────────────────
def prompt_keyword_coverage(prompt_clean, response_clean):
    """Fraction of prompt keywords (non-stopword) present in response."""
    p_words = set(prompt_clean.split())
    r_words = set(response_clean.split())
    if not p_words: return 0.0
    return len(p_words & r_words) / len(p_words)

def prompt_question_count(raw_prompt):
    return raw_prompt.count('?')

def prompt_question_type(raw_prompt):
    """
    Encode prompt type as int:
    0=other, 1=how, 2=why, 3=what, 4=write/create, 5=explain, 6=compare/vs, 7=list
    """
    p = raw_prompt.lower().strip()
    if re.search(r'\b(compare|vs|versus|difference between)\b', p): return 6
    if re.search(r'\b(list|enumerate|give me)\b', p): return 7
    if re.search(r'\b(write|create|generate|draft|make)\b', p): return 4
    if re.search(r'\bexplain\b', p): return 5
    if p.startswith('how'): return 1
    if p.startswith('why'): return 2
    if p.startswith('what'): return 3
    return 0

def has_code_in_prompt(raw_prompt):
    return int('```' in raw_prompt or bool(re.search(r'`[^`]+`', raw_prompt)))

def named_entity_overlap(prompt_raw, response_raw):
    """Fraction of prompt named entities that appear in response. Requires spaCy."""
    if not HAS_SPACY: return 0.0
    try:
        p_ents = {e.text.lower() for e in _nlp(prompt_raw[:500]).ents}
        r_ents = {e.text.lower() for e in _nlp(response_raw[:500]).ents}
        if not p_ents: return 0.0
        return len(p_ents & r_ents) / len(p_ents)
    except:
        return 0.0

def trigram_overlap(a, b):
    """Trigram Jaccard overlap between two cleaned texts."""
    def tg(t):
        tok = t.split()
        return set(zip(tok[:-2], tok[1:-1], tok[2:])) if len(tok) >= 3 else set()
    ta, tb = tg(a), tg(b)
    if not ta or not tb: return 0.0
    return len(ta & tb) / (len(ta | tb) + 1e-9)

# ── NEW: Lexical specificity helpers ──────────────────────────
def number_count(text):
    return len(re.findall(r'\b\d+\.?\d*\b', text))

def rare_word_ratio(clean_text_tokens):
    """Fraction of words NOT in common 5000 words list."""
    tok = [w for w in clean_text_tokens.split() if len(w) >= 2]
    if not tok: return 0.0
    return sum(1 for w in tok if w not in COMMON_5000_WORDS) / len(tok)

def unique_bigram_ratio(clean_text_tokens):
    tok = clean_text_tokens.split()
    if len(tok) < 2: return 0.0
    bigrams = list(zip(tok[:-1], tok[1:]))
    return len(set(bigrams)) / len(bigrams)

# ── NEW: Cross-response differential helpers ──────────────────
def unique_content_ratio(response_clean, other_clean):
    """Fraction of response's words that are unique to it (not in other)."""
    r_words = response_clean.split()
    o_words = set(other_clean.split())
    if not r_words: return 0.0
    return sum(1 for w in r_words if w not in o_words) / len(r_words)

def keyword_coverage_of_other(a_clean, b_clean):
    """Fraction of B's keywords that appear in A (A's dominance over B)."""
    b_words = set(b_clean.split())
    a_words = set(a_clean.split())
    if not b_words: return 1.0
    return len(b_words & a_words) / len(b_words)

def cosine_sim_tfidf(a, b, vectorizer=None):
    """TF-IDF cosine similarity between two texts (computed per-pair)."""
    if not a.strip() or not b.strip(): return 0.0
    try:
        from sklearn.feature_extraction.text import TfidfVectorizer as TV
        from sklearn.metrics.pairwise import cosine_similarity
        v = TV(max_features=500).fit_transform([a, b])
        return float(cosine_similarity(v[0], v[1])[0, 0])
    except:
        return 0.0

# ── VERBOSITY non-linear helpers ───────────────────────────────
def log_word_count(text):
    return np.log1p(len(text.split()))

def len_ratio_squared(len_a, len_b):
    ratio = len_a / (len_b + 1)
    return ratio ** 2

def extreme_verbosity_flag(len_a, len_b, threshold=3.0):
    """1 if A is > threshold times longer than B."""
    return int(len_a > threshold * (len_b + 1))

def prompt_to_response_ratio(len_prompt, len_response):
    return len_response / (len_prompt + 1)

## Load processed data

From Notebook 02: `data/df_processed.csv`.

In [ ]:
df = pd.read_csv('../data/df_processed.csv')
print(f"Loaded {len(df):,} rows × {df.shape[1]} columns")

required = (
    'prompt_text', 'response_a_text', 'response_b_text',
    'prompt_clean', 'response_a_clean', 'response_b_clean',
    'text_full', 'text_diff', 'label',
)
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"df_processed.csv missing columns {missing}; run Notebook 02 first.")


## Stage 4 — same block as `train.py`

Builds TF-IDF, all dense frames, `StandardScaler(with_mean=False)`, `hstack`, writes `X_combined.npz`, pickles, `dense_features.csv`.

In [ ]:
# STAGE 4: FEATURE ENGINEERING (mirrors scripts/train.py)

# 4A: TF-IDF
print("  [4A] TF-IDF...")
tfidf_full = TfidfVectorizer(ngram_range=(1,2), max_features=30000, sublinear_tf=True, min_df=3, max_df=0.95)
tfidf_diff = TfidfVectorizer(ngram_range=(1,2), max_features=15000, sublinear_tf=True, min_df=2, max_df=0.95)
X_tfidf_full = tfidf_full.fit_transform(df['text_full'])
X_tfidf_diff = tfidf_diff.fit_transform(df['text_diff'])
print(f"  tfidf_full={X_tfidf_full.shape}  tfidf_diff={X_tfidf_diff.shape}")

# 4B: Original numeric features
print("  [4B] Original handcrafted numeric features...")
ra, rb = df['response_a_text'].tolist(), df['response_b_text'].tolist()
ra_c, rb_c, pa_c = df['response_a_clean'].tolist(), df['response_b_clean'].tolist(), df['prompt_clean'].tolist()
len_a  = np.array([len(x.split()) for x in ra])
len_b  = np.array([len(x.split()) for x in rb])
len_p  = np.array([len(x.split()) for x in df['prompt_text']])
vr_a   = np.array([vocab_richness(x) for x in ra_c])
vr_b   = np.array([vocab_richness(x) for x in rb_c])
sc_a   = np.array([count_sentences(x) for x in ra])
sc_b   = np.array([count_sentences(x) for x in rb])

orig_numeric = pd.DataFrame({
    'len_a': len_a, 'len_b': len_b, 'len_prompt': len_p,
    'len_ratio_a_b': len_a / (len_b + 1),
    'len_diff_abs': np.abs(len_a - len_b),
    'longer_is_a': (len_a > len_b).astype(int),
    'vocab_rich_a': vr_a, 'vocab_rich_b': vr_b, 'vocab_rich_diff': vr_a - vr_b,
    'sentences_a': sc_a, 'sentences_b': sc_b, 'sentences_diff': sc_a - sc_b,
    'has_code_a':    [has_code(x)    for x in ra],
    'has_code_b':    [has_code(x)    for x in rb],
    'has_bullets_a': [has_bullets(x) for x in ra],
    'has_bullets_b': [has_bullets(x) for x in rb],
    'has_headers_a': [has_headers(x) for x in ra],
    'has_headers_b': [has_headers(x) for x in rb],
})
print(f"  Original features: {orig_numeric.shape[1]} columns")

# 4C: Original similarity features
print("  [4C] Original lexical similarity features...")
orig_similarity = pd.DataFrame({
    'jaccard_a_b':        [jaccard(a, b)       for a, b in zip(ra_c, rb_c)],
    'overlap_a_b':        [overlap_coeff(a, b) for a, b in zip(ra_c, rb_c)],
    'bigram_overlap_a_b': [bigram_overlap(a, b) for a, b in zip(ra_c, rb_c)],
    'jaccard_prompt_a':   [jaccard(p, a)       for p, a in zip(pa_c, ra_c)],
    'jaccard_prompt_b':   [jaccard(p, b)       for p, b in zip(pa_c, rb_c)],
})

# ── NEW FEATURE GROUPS ─────────────────────────────────────────

# 4D-NEW: Readability features
print("  [4D] NEW — Readability features...")
fk_a = np.array([fk_grade(x) for x in ra])
fk_b = np.array([fk_grade(x) for x in rb])
fe_a = np.array([flesch_ease(x) for x in ra])
fe_b = np.array([flesch_ease(x) for x in rb])
syll_a = np.array([avg_syllables_per_word(x) for x in ra])
syll_b = np.array([avg_syllables_per_word(x) for x in rb])
avg_wps_a = np.array([avg_words_per_sentence(x) for x in ra])
avg_wps_b = np.array([avg_words_per_sentence(x) for x in rb])
read_features = pd.DataFrame({
    'fk_grade_a':          fk_a,
    'fk_grade_b':          fk_b,
    'fk_grade_diff':       fk_a - fk_b,
    'flesch_ease_a':       fe_a,
    'flesch_ease_b':       fe_b,
    'flesch_ease_diff':    fe_a - fe_b,
    'avg_syll_word_a':     syll_a,
    'avg_syll_word_b':     syll_b,
    'avg_syll_word_diff':  syll_a - syll_b,
    'avg_words_sent_a':    avg_wps_a,
    'avg_words_sent_b':    avg_wps_b,
    'avg_words_sent_diff': avg_wps_a - avg_wps_b,
})
print(f"  Readability: {read_features.shape[1]} columns")

# 4E-NEW: Structural formatting depth
print("  [4E] NEW — Structural formatting depth...")
struct_features = pd.DataFrame({
    'header_count_a':           [count_headers(x)     for x in ra],
    'header_count_b':           [count_headers(x)     for x in rb],
    'header_count_diff':        [count_headers(a) - count_headers(b) for a, b in zip(ra, rb)],
    'bullet_count_a':           [count_bullets(x)     for x in ra],
    'bullet_count_b':           [count_bullets(x)     for x in rb],
    'bullet_count_diff':        [count_bullets(a) - count_bullets(b) for a, b in zip(ra, rb)],
    'code_block_count_a':       [count_code_blocks(x) for x in ra],
    'code_block_count_b':       [count_code_blocks(x) for x in rb],
    'code_block_count_diff':    [count_code_blocks(a) - count_code_blocks(b) for a, b in zip(ra, rb)],
    'paragraph_count_a':        [count_paragraphs(x)  for x in ra],
    'paragraph_count_b':        [count_paragraphs(x)  for x in rb],
    'paragraph_count_diff':     [count_paragraphs(a) - count_paragraphs(b) for a, b in zip(ra, rb)],
    'formatting_richness_a':    [formatting_richness(x) for x in ra],
    'formatting_richness_b':    [formatting_richness(x) for x in rb],
    'formatting_richness_diff': [formatting_richness(a) - formatting_richness(b) for a, b in zip(ra, rb)],
})
print(f"  Formatting depth: {struct_features.shape[1]} columns")

# 4F-NEW: Sentiment / tone
print("  [4F] NEW — Sentiment and tone features...")
sent_a = np.array([sentiment_polarity(x) for x in ra])
sent_b = np.array([sentiment_polarity(x) for x in rb])
pos_a = np.array([positive_word_ratio(x) for x in ra_c])
pos_b = np.array([positive_word_ratio(x) for x in rb_c])
neg_a = np.array([negative_word_ratio(x) for x in ra_c])
neg_b = np.array([negative_word_ratio(x) for x in rb_c])
hedge_a = np.array([hedge_word_ratio(x) for x in ra_c])
hedge_b = np.array([hedge_word_ratio(x) for x in rb_c])
syco_a = np.array([sycophancy_flag(x) for x in ra])
syco_b = np.array([sycophancy_flag(x) for x in rb])
sent_features = pd.DataFrame({
    'sentiment_a':       sent_a,
    'sentiment_b':       sent_b,
    'sentiment_diff':    sent_a - sent_b,
    'pos_word_ratio_a':  pos_a,
    'pos_word_ratio_b':  pos_b,
    'pos_word_diff':     pos_a - pos_b,
    'neg_word_ratio_a':  neg_a,
    'neg_word_ratio_b':  neg_b,
    'neg_word_diff':     neg_a - neg_b,
    'hedge_ratio_a':     hedge_a,
    'hedge_ratio_b':     hedge_b,
    'hedge_ratio_diff':  hedge_a - hedge_b,
    'sycophancy_a':      syco_a,
    'sycophancy_b':      syco_b,
    'sycophancy_diff':   syco_a - syco_b,
})
print(f"  Sentiment/tone: {sent_features.shape[1]} columns")

# 4G-NEW: Prompt-response alignment
print("  [4G] NEW — Prompt alignment features...")
pt_list = df['prompt_text'].tolist()
prompt_q_type = np.array([prompt_question_type(x) for x in pt_list])

print("    -> Extracting NER using SpaCy (batched for speed)...")
if HAS_SPACY:
    p_ents_list = [{e.text.lower() for e in doc.ents} for doc in _nlp.pipe([p[:500] for p in pt_list], n_process=-1, batch_size=1000)]
    ra_ents_list = [{e.text.lower() for e in doc.ents} for doc in _nlp.pipe([r[:500] for r in ra], n_process=-1, batch_size=1000)]
    rb_ents_list = [{e.text.lower() for e in doc.ents} for doc in _nlp.pipe([r[:500] for r in rb], n_process=-1, batch_size=1000)]
else:
    p_ents_list = [set()] * len(pt_list)
    ra_ents_list = [set()] * len(ra)
    rb_ents_list = [set()] * len(rb)

def _calc_ner(p_ents, r_ents):
    if not p_ents: return 0.0
    return len(p_ents & r_ents) / len(p_ents)

ner_ov_a = [_calc_ner(p, r) for p, r in zip(p_ents_list, ra_ents_list)]
ner_ov_b = [_calc_ner(p, r) for p, r in zip(p_ents_list, rb_ents_list)]

align_features = pd.DataFrame({
    'prompt_kw_coverage_a':   [prompt_keyword_coverage(p, a) for p, a in zip(pa_c, ra_c)],
    'prompt_kw_coverage_b':   [prompt_keyword_coverage(p, b) for p, b in zip(pa_c, rb_c)],
    'prompt_kw_cov_diff':     [prompt_keyword_coverage(p, a) - prompt_keyword_coverage(p, b)
                               for p, a, b in zip(pa_c, ra_c, rb_c)],
    'prompt_question_count':  [prompt_question_count(x) for x in pt_list],
    'prompt_question_type':   prompt_q_type,
    'prompt_has_code':        [has_code_in_prompt(x) for x in pt_list],
    'trigram_overlap_a_b':    [trigram_overlap(a, b) for a, b in zip(ra_c, rb_c)],
    'trigram_overlap_pa':     [trigram_overlap(p, a) for p, a in zip(pa_c, ra_c)],
    'trigram_overlap_pb':     [trigram_overlap(p, b) for p, b in zip(pa_c, rb_c)],
    'ner_overlap_a':          ner_ov_a,
    'ner_overlap_b':          ner_ov_b,
    'ner_overlap_diff':       [a - b for a, b in zip(ner_ov_a, ner_ov_b)],
})
print(f"  Prompt alignment: {align_features.shape[1]} columns")

# 4H-NEW: Lexical specificity
print("  [4H] NEW — Lexical specificity features...")
spec_features = pd.DataFrame({
    'number_count_a':        [number_count(x) for x in ra],
    'number_count_b':        [number_count(x) for x in rb],
    'number_count_diff':     [number_count(a) - number_count(b) for a, b in zip(ra, rb)],
    'rare_word_ratio_a':     [rare_word_ratio(x) for x in ra_c],
    'rare_word_ratio_b':     [rare_word_ratio(x) for x in rb_c],
    'rare_word_ratio_diff':  [rare_word_ratio(a) - rare_word_ratio(b) for a, b in zip(ra_c, rb_c)],
    'uniq_bigram_ratio_a':   [unique_bigram_ratio(x) for x in ra_c],
    'uniq_bigram_ratio_b':   [unique_bigram_ratio(x) for x in rb_c],
    'uniq_bigram_ratio_diff':[unique_bigram_ratio(a) - unique_bigram_ratio(b) for a, b in zip(ra_c, rb_c)],
    'specificity_score_a':   [number_count(a) / (len(ra_c[i].split()) + 1) + rare_word_ratio(ra_c[i])
                              for i, a in enumerate(ra)],
    'specificity_score_b':   [number_count(b) / (len(rb_c[i].split()) + 1) + rare_word_ratio(rb_c[i])
                              for i, b in enumerate(rb)],
    'specificity_diff':      [
        (number_count(ra[i]) / (len(ra_c[i].split()) + 1) + rare_word_ratio(ra_c[i]))
        - (number_count(rb[i]) / (len(rb_c[i].split()) + 1) + rare_word_ratio(rb_c[i]))
        for i in range(len(ra))
    ],
})
print(f"  Lexical specificity: {spec_features.shape[1]} columns")

# 4I-NEW: Verbosity non-linear signals
print("  [4I] NEW — Verbosity non-linear features...")
verb_features = pd.DataFrame({
    'log_len_a':              np.log1p(len_a),
    'log_len_b':              np.log1p(len_b),
    'log_len_diff':           np.log1p(len_a) - np.log1p(len_b),
    'len_ratio_squared':      len_ratio_squared(len_a, len_b),
    'extreme_verb_a':         [extreme_verbosity_flag(a, b) for a, b in zip(len_a, len_b)],
    'extreme_verb_b':         [extreme_verbosity_flag(b, a) for a, b in zip(len_a, len_b)],
    'prompt_to_resp_ratio_a': prompt_to_response_ratio(len_p, len_a),
    'prompt_to_resp_ratio_b': prompt_to_response_ratio(len_p, len_b),
    'prompt_to_resp_diff':    prompt_to_response_ratio(len_p, len_a) - prompt_to_response_ratio(len_p, len_b),
})
print(f"  Verbosity non-linear: {verb_features.shape[1]} columns")

# 4J-NEW: Cross-response differential features
print("  [4J] NEW — Cross-response differential features...")
cross_features = pd.DataFrame({
    'unique_content_ratio_a': [unique_content_ratio(a, b) for a, b in zip(ra_c, rb_c)],
    'unique_content_ratio_b': [unique_content_ratio(b, a) for a, b in zip(ra_c, rb_c)],
    'a_covers_b_keywords':    [keyword_coverage_of_other(a, b) for a, b in zip(ra_c, rb_c)],
    'b_covers_a_keywords':    [keyword_coverage_of_other(b, a) for a, b in zip(ra_c, rb_c)],
    'response_divergence':    [1.0 - jaccard(a, b) for a, b in zip(ra_c, rb_c)],
})
print(f"  Cross-response diff: {cross_features.shape[1]} columns")

# 4K-NEW: Prompt complexity features
print("  [4K] NEW — Prompt complexity features...")
prompt_comp = pd.DataFrame({
    'prompt_word_count':      len_p,
    'log_prompt_len':         np.log1p(len_p),
    'prompt_sent_count':      np.array([count_sentences(x) for x in pt_list]),
    'prompt_q_count':         np.array([prompt_question_count(x) for x in pt_list]),
    # NOTE: prompt_question_type is intentionally omitted here —
    # it is already included in align_features (4G). Including it
    # again would cause pd.concat to silently produce a duplicate
    # column, creating a 109-feature scaler vs 108-feature inference mismatch.
    'prompt_complexity_score':len_p * np.array([prompt_question_count(x) + 1 for x in pt_list]),
    'prompt_multi_sent':      (np.array([count_sentences(x) for x in pt_list]) > 1).astype(int),
})
print(f"  Prompt complexity: {prompt_comp.shape[1]} columns")

# ── 4L: Combine ALL dense features ────────────────────────────
print("  [4L] Combining all dense features...")
all_dense = pd.concat([
    orig_numeric,
    orig_similarity,
    read_features,
    struct_features,
    sent_features,
    align_features,
    spec_features,
    verb_features,
    cross_features,
    prompt_comp,
], axis=1).fillna(0)

print(f"  Total dense features: {all_dense.shape[1]}")
print(f"    Original: {orig_numeric.shape[1] + orig_similarity.shape[1]}")
print(f"    Readability: {read_features.shape[1]}")
print(f"    Formatting depth: {struct_features.shape[1]}")
print(f"    Sentiment/tone: {sent_features.shape[1]}")
print(f"    Prompt alignment: {align_features.shape[1]}")
print(f"    Lexical specificity: {spec_features.shape[1]}")
print(f"    Verbosity non-linear: {verb_features.shape[1]}")
print(f"    Cross-response diff: {cross_features.shape[1]}")
print(f"    Prompt complexity: {prompt_comp.shape[1]}")

scaler = StandardScaler(with_mean=False)
X_dense = sp.csr_matrix(scaler.fit_transform(all_dense.values.astype(np.float32)))
X = sp.hstack([X_tfidf_full, X_tfidf_diff, X_dense], format='csr')
X_tfidf_only = X[:, :X_tfidf_full.shape[1] + X_tfidf_diff.shape[1]]
y = df['label'].values

print(f"  Final matrix: {X.shape}  Memory: {X.data.nbytes/1e6:.1f} MB")

# Save artifacts
sp.save_npz(f'{ARTIFACTS}/X_combined.npz', X)
np.save(f'{ARTIFACTS}/y_labels.npy', y)
joblib.dump(tfidf_full, f'{ARTIFACTS}/tfidf_full.pkl')
joblib.dump(tfidf_diff, f'{ARTIFACTS}/tfidf_diff.pkl')
joblib.dump(scaler,     f'{ARTIFACTS}/scaler.pkl')
all_dense.to_csv(f'{ARTIFACTS}/dense_features.csv', index=False)
print(f"  Artifacts saved to {ARTIFACTS}/")


